# Packages

In [ ]:
%pip install /kaggle/input/lmsys-packages/triton-2.2.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

In [ ]:
%pip install /kaggle/input/lmsys-packages/xformers-0.0.24042abc8.d20240802-cp310-cp310-linux_x86_64.whl

In [ ]:
!cp -r /kaggle/input/lmsys-modules-0805 human_pref

In [ ]:
%%writefile human_pref/data/processors.py
import json

import torch


class ProcessorPAB:
    # PROMPT_PREFIX = """You will be given two responses from two Assistants to the same Prompt. 
    # Your task is to decide which response is better based on its relevance, helpfulness, and level of detail. 
    # Begin your evaluation by comparing the two responses and provide a short explanation. Do not favor certain names of
    # the assistants. Be as objective as possible. After providing your explanation, output your final verdict by strictly 
    # following this format with no tie : "[[A]]" if assistant A is better, "[[B]]" if assistant B is better."""
    
    PROMPT_PREFIX ="""Act as an impartial judge and evaluate the quality of the responses provided by two AI assistants to the user question displayed below. You should choose the assistant that
    follows the user’s instructions and answers the user’s question better. Your evaluation should consider factors such as the helpfulness, relevance, accuracy, depth, creativity, topic coverage
    and level of detail of their responses. Begin your evaluation by comparing the two responses and provide explanation and observations about each responses then provide reason why would you prefer certain
    response over other response. Avoid any position biases and ensure that the order in which the responses were presented does not influence your decision. Do not allow the length of the responses to influence your evaluation. 
    Do not favor certain names of the assistants. Be as objective as possible. After providing your explanation and observation, output your final verdict by strictly following this format: "[[A]]" if assistant A is better, "[[B]]" if assistant B is better."""

#     PROMPT_PREFIX = """
#     Please act as an impartial judge and evaluate the quality of the responses provided by two
#     AI assistants to the user question displayed below. You should choose the assistant that
#     follows the user’s instructions and answers the user’s question better.
#     Carefully analyze response from Assistant A and Assistant B in relation to the User's Query. Determine which response is more likely to be selected by a user based on the following criteria:
#     - Completeness in addressing the query
#     - Accuracy of information
#     - Clarity and coherence
#     - Conciseness vs appropriate detail
#     - Helpful examples or explanations when needed
#     - Professional yet engaging tone
#     - Sound reasoning and logic
#     - Format and presentation
#     Avoid any position biases and ensure that the order in which the responses were presented does not influence your decision.
#     Do not favor certain names of the assistants. Be as objective as possible. After providing your explanation, output your final verdict by strictly following this format: 
#     "[[A]]" if assistant A is better, 
#     "[[B]]" if assistant B is better.
# """

    PROMPT_SUFFIX = "verdict is: [["

    LABEL_COLS = ["winner_model_a", "winner_model_b", "winner_tie"]

    def __init__(self, tokenizer, max_length, support_system_role):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.support_system_role = support_system_role

    def build_conversation(self, prompts, responses_a, responses_b):
        head = "<|The Start of Conversation between a User and two Assistants|>"
        tail = "<|The End of Conversation between a User and two Assistants|>\n"
        parts = []
        for prompt, response_a, response_b in zip(prompts, responses_a, responses_b):
            if prompt is None:
                prompt = "null"
            if response_a is None:
                response_a = "null"
            if response_b is None:
                response_b = "null"
            parts.append(
                f"\n### User:\n{prompt}\n\n### Assistant A:\n{response_a}\n\n### Assistant B:\n{response_b}\n"
            )
        text = "".join(parts)
        input_ids = self.tokenizer(
            text,
            add_special_tokens=False,
            max_length=self.max_length,
            truncation=True,
        ).input_ids

        truncated_text = self.tokenizer.decode(input_ids)
        return head + truncated_text + tail

    def build_input(self, data):
        conversation = self.build_conversation(
            [data["prompt"]],
            [data["response_a"]],
            [data["response_b"]],
        )
        if self.support_system_role:
            messages = [
                {"role": "system", "content": self.PROMPT_PREFIX},
                {"role": "user", "content": conversation},
            ]
        else:
            messages = [
                {"role": "user", "content": f"{self.PROMPT_PREFIX}\n{conversation}"},
            ]
        input_text = (
            self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            + self.PROMPT_SUFFIX
        )
        input_ids = self.tokenizer(
            input_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).input_ids[0]
        label = torch.tensor([data[col] for col in self.LABEL_COLS]).float()
        return dict(
            input_ids=input_ids,
            input_text=input_text,
            label=label,
        )

# Prepare test file

In [ ]:
# %%writefile prepare_test_file.py
import pandas as pd
import os
df = pd.read_parquet("/kaggle/input/wsdm-cup-multilingual-chatbot-arena/test.parquet")
df["winner_model_a"] = 1
df["winner_model_b"] = 0
df["winner_tie"] = 0
df.to_parquet("test.parquet", index=False)
# df["response_a"], df["response_b"] = df["response_b"], df["response_a"]
# df.to_parquet("test_swap.parquet", index=False)

In [ ]:
# !python prepare_test_file.py

# Inference: gemma2-9b

In [ ]:
%%writefile predict_m0.py
import time
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoTokenizer

# from xformers.ops.fmha.attn_bias import BlockDiagonalCausalMask
from human_pref.models.modeling_gemma2 import Gemma2ForSequenceClassification
from human_pref.data.processors import ProcessorPAB
from human_pref.data.dataset import LMSYSDataset
from human_pref.data.collators import VarlenCollator, ShardedMaxTokensCollator
from human_pref.utils import to_device


model_name_or_path = "/kaggle/input/lmsys-checkpoints-0-0805"
csv_path = "test.parquet"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
processor = ProcessorPAB(
    tokenizer=tokenizer,
    max_length=4096,
    support_system_role=False,
)
dataset = LMSYSDataset(
    csv_file=csv_path,
    query=None,
    processor=processor,
    include_swap=False,
    is_parquet=True,
)
dataloader = DataLoader(
    dataset,
    batch_size=80,
    num_workers=4,
    collate_fn=ShardedMaxTokensCollator(
        max_tokens=8192, base_collator=VarlenCollator()
    ),
)

# model for pipelined inference
num_hidden_layers = 42
device_map = {
    "model.embed_tokens": "cuda:0",
    "model.norm": "cuda:1",
    "score": "cuda:1",
}
for i in range(num_hidden_layers // 2):
    device_map[f"model.layers.{i}"] = "cuda:0"
for i in range(num_hidden_layers // 2, num_hidden_layers):
    device_map[f"model.layers.{i}"] = "cuda:1"

model = Gemma2ForSequenceClassification.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.float16,
    device_map=device_map,
)

# inv_freq clones for each device
config = model.config
dim = config.head_dim
inv_freq = 1.0 / (
    config.rope_theta ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim)
)
inv_freq0 = inv_freq.to("cuda:0")
inv_freq1 = inv_freq.to("cuda:1")


# for name, p in model.named_parameters():
#     print(name, p.device)
# for name, b in model.model.named_buffers():
#     print(name, b.device)

# pipeline parallelism with two GPUs
t1 = time.time()
is_first = True
hidden_states = None
outs = []
for batch in tqdm(dataloader):
    for micro_batch in batch:
        input_ids = to_device(micro_batch["input_ids"], "cuda:0")
        seq_info = dict(
            cu_seqlens=micro_batch["cu_seqlens"],
            position_ids=micro_batch["position_ids"],
            max_seq_len=micro_batch["max_seq_len"],
            # attn_bias=BlockDiagonalCausalMask.from_seqlens(micro_batch["seq_lens"]),
        )
        seq_info = to_device(seq_info, "cuda:0")
        if is_first:
            with torch.no_grad(), torch.cuda.amp.autocast():
                prev_hidden_states = model.forward_part1(input_ids, seq_info, inv_freq0)
            is_first = False
            prev_seq_info, prev_hidden_states = to_device(
                [seq_info, prev_hidden_states], "cuda:1"
            )
            continue
        with torch.no_grad(), torch.cuda.amp.autocast():
            logits = model.forward_part2(prev_hidden_states, prev_seq_info, inv_freq1)
            hidden_states = model.forward_part1(input_ids, seq_info, inv_freq0)

            prev_seq_info, prev_hidden_states = to_device(
                [seq_info, hidden_states], "cuda:1"
            )
            outs.append(logits.cpu())

# last micro-batch
with torch.no_grad(), torch.cuda.amp.autocast():
    logits = model.forward_part2(prev_hidden_states, prev_seq_info, inv_freq1)
    outs.append(logits.cpu())

pred = torch.cat(outs, dim=0)
prob = pred.softmax(-1)
print(dataset.evaluate(prob.numpy()))

np.save('prob_m0.npy', prob)

t2 = time.time()
print(f"elapsed time: {t2 - t1}s")
print(f"elapsed time 10k: {(t2 - t1) * 100 / 3600}h")
print(f"elapsed time 25k: {(t2 - t1) * 250 / 3600}h")

In [ ]:
!python predict_m0.py

# Inference: llama3-8b

In [ ]:
%%writefile predict_m3.py
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoTokenizer

from xformers.ops.fmha.attn_bias import BlockDiagonalCausalMask
from human_pref.models.modeling_llama import LlamaForSequenceClassification
from human_pref.data.processors import ProcessorPAB
from human_pref.data.dataset import LMSYSDataset
from human_pref.data.collators import VarlenCollator, ShardedMaxTokensCollator
from human_pref.utils import to_device


model_name_or_path = "/kaggle/input/lmsys-checkpoints-3-0805"
csv_path = "test_swap.parquet"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
tokenizer.deprecation_warnings[
    "sequence-length-is-longer-than-the-specified-maximum"
] = True
processor = ProcessorPAB(
    tokenizer=tokenizer,
    max_length=4096,
    support_system_role=True,
)
dataset = LMSYSDataset(
    csv_file=csv_path,
    query=None,
    processor=processor,
    include_swap=False,
    is_parquet=True,
)
dataloader = DataLoader(
    dataset,
    batch_size=80,
    num_workers=4,
    collate_fn=ShardedMaxTokensCollator(
        max_tokens=8192, base_collator=VarlenCollator()
    ),
)

# model for pipelined inference
num_hidden_layers = 32
device_map = {
    "model.embed_tokens": "cuda:0",
    "model.norm": "cuda:1",
    "score": "cuda:1",
}
for i in range(num_hidden_layers // 2):
    device_map[f"model.layers.{i}"] = "cuda:0"
for i in range(num_hidden_layers // 2, num_hidden_layers):
    device_map[f"model.layers.{i}"] = "cuda:1"

model = LlamaForSequenceClassification.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.float16,
    device_map=device_map,
)

# inv_freq clones for each device
config = model.config
dim = config.hidden_size // config.num_attention_heads
inv_freq = 1.0 / (
    config.rope_theta ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim)
)
inv_freq0 = inv_freq.to("cuda:0")
inv_freq1 = inv_freq.to("cuda:1")


# for name, p in model.named_parameters():
#     print(name, p.device)
# for name, b in model.model.named_buffers():
#     print(name, b.device)

# pipeline parallelism with two GPUs
is_first = True
hidden_states = None
outs = []
for batch in tqdm(dataloader):
    for micro_batch in batch:
        input_ids = to_device(micro_batch["input_ids"], "cuda:0")
        seq_info = dict(
            cu_seqlens=micro_batch["cu_seqlens"],
            position_ids=micro_batch["position_ids"],
            max_seq_len=micro_batch["max_seq_len"],
            attn_bias=BlockDiagonalCausalMask.from_seqlens(micro_batch["seq_lens"]),
        )
        seq_info = to_device(seq_info, "cuda:0")
        if is_first:
            with torch.no_grad(), torch.cuda.amp.autocast():
                prev_hidden_states = model.forward_part1(input_ids, seq_info, inv_freq0)
            is_first = False
            prev_seq_info, prev_hidden_states = to_device(
                [seq_info, prev_hidden_states], "cuda:1"
            )
            continue
        with torch.no_grad(), torch.cuda.amp.autocast():
            logits = model.forward_part2(prev_hidden_states, prev_seq_info, inv_freq1)
            hidden_states = model.forward_part1(input_ids, seq_info, inv_freq0)

            prev_seq_info, prev_hidden_states = to_device(
                [seq_info, hidden_states], "cuda:1"
            )
            outs.append(logits.cpu())

# last micro-batch
with torch.no_grad(), torch.cuda.amp.autocast():
    logits = model.forward_part2(prev_hidden_states, prev_seq_info, inv_freq1)
    outs.append(logits.cpu())


pred = torch.cat(outs, dim=0)
prob = pred.softmax(-1)
print(dataset.evaluate(prob.numpy()))

np.save('prob_m3.npy', prob)

In [ ]:
# !python predict_m3.py

## TFIDF - Predictions

In [ ]:
import pandas as pd , numpy as np
import os
from sklearn.model_selection import (train_test_split, cross_val_score, StratifiedKFold)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import plotly.express as px, seaborn as sns, matplotlib.pyplot as plt
sns.set_style('darkgrid')
from sklearn.metrics import make_scorer, accuracy_score
from sklearn.metrics import log_loss

import xgboost
import optuna
from sklearn.ensemble import  VotingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import optuna
from sklearn.metrics import accuracy_score

import warnings
warnings.simplefilter('ignore')

In [ ]:
train = pd.read_parquet('/kaggle/input/wsdm-cup-multilingual-chatbot-arena/train.parquet')
test = pd.read_parquet('/kaggle/input/wsdm-cup-multilingual-chatbot-arena/test.parquet')

In [ ]:
train = train[['prompt','response_a','response_b','winner']]
train_5 = pd.read_parquet('/kaggle/input/arena-hard-auto-dataset/gpt_4.11_eval_model_arena-hard-auto.parquet')
train_5= train_5[train_5['winner'] != 'tie']
del train_5['question_id']
train = pd.concat([train,train_5])
train = train.drop_duplicates()
train = train.dropna()
train = train.sample(frac=1).reset_index(drop=True)

In [ ]:
train,valid=train_test_split(train, test_size=0.15, stratify=train["winner"],random_state=42)

# Train set can be inverted (and winner too) to get twice the data from the available training dataset
train_inv=train.copy()
train_inv["response_a"],train_inv["response_b"]=train_inv["response_b"],train_inv["response_a"]
train_inv["winner"]=train_inv["winner"].apply(lambda x: "model_a" if "b" in x else "model_b")

test_inv=test.copy()
test_inv["response_a"],test_inv["response_b"]=test_inv["response_b"],test_inv["response_a"]

In [ ]:

X = train[['prompt', 'response_a', 'response_b']]

# Initialize a single TfidfVectorizer
tfidf = TfidfVectorizer(max_features=300)

# Combine all text data to fit the vectorizer
all_text = pd.concat([X['prompt'], X['response_a'], X['response_b']])

# Fit the vectorizer on all text data
tfidf.fit(all_text)

In [ ]:
# Here I compute some features
def compute_feats(dataf):
    df = dataf.copy()
    feats = []
    for col in ["response_a","response_b","prompt"]:
        # response lenght is a key factor when choosing between two responses
        df[f"{col}_len"]=df[f"{col}"].str.len()
    
        # Some characters counting features 
        df[f"{col}_spaces"]=df[f"{col}"].str.count("\s")
        df[f"{col}_punct"]=df[f"{col}"].str.count(",|\.|!")
        df[f"{col}_question_mark"]=df[f"{col}"].str.count("\?")
        df[f"{col}_quot"]=df[f"{col}"].str.count("'|\"")
        df[f"{col}_formatting_chars"]=df[f"{col}"].str.count("\*|\_")
        df[f"{col}_math_chars"]=df[f"{col}"].str.count("\-|\+|\=")
        df[f"{col}_curly_open"]=df[f"{col}"].str.count("\{")
        df[f"{col}_curly_close"]=df[f"{col}"].str.count("}")
        df[f"{col}_round_open"]=df[f"{col}"].str.count("\(")
        df[f"{col}_round_close"]=df[f"{col}"].str.count("\)")
        df[f"{col}_accent_chars"]=df[f"{col}"].str.count("è|ò|à|ù|é|ì")
        df[f"{col}_special_chars"]=df[f"{col}"].str.count("\W")
        df[f"{col}_digits"]=df[f"{col}"].str.count("\d")/df[f"{col}_len"]
        df[f"{col}_lower"]=df[f"{col}"].str.count("[a-z]").astype("float32")/df[f"{col}_len"]
        df[f"{col}_upper"]=df[f"{col}"].str.count("[A-Z]").astype("float32")/df[f"{col}_len"]
        df[f"{col}_chinese"]=df[f"{col}"].str.count(r'[\u4e00-\u9fff]+').astype("float32")/df[f"{col}_len"]
        df[f"{col}_tild"]=df[f"{col}"].str.count("~")>0
        df[f"{col}_letters"] = df[f"{col}"].str.count("[a-zA-Z]")

        # Feature that show how balanced are curly and round brackets
        df[f"{col}_round_balance"]=df[f"{col}_round_open"]-df[f"{col}_round_close"]
        df[f"{col}_curly_balance"]=df[f"{col}_curly_open"]-df[f"{col}_curly_close"]

        # Feature that tells if the string json is present somewhere (e.g. asking a json response or similar)
        df[f"{col}_json"]=df[f"{col}"].str.lower().str.count("json")
        df[f"{col}_yaml"]=df[f"{col}"].str.lower().str.count("yaml")
        
        feats = feats + [f"{col}_len",f"{col}_spaces",f"{col}_punct", f"{col}_question_mark",f"{col}_quot", 
                        f"{col}_formatting_chars",f"{col}_math_chars",f"{col}_curly_open",f"{col}_curly_close",f"{col}_round_open",  
                        f"{col}_round_close",f"{col}_accent_chars",f"{col}_special_chars",f"{col}_digits",f"{col}_lower",  
                        f"{col}_upper",f"{col}_chinese",f"{col}_tild",f"{col}_letters"
                        ]
    df["response_a_prompt_overlap"] = df.apply(
        lambda x: len(set(x["prompt"].split()).intersection(set(x["response_a"].split()))) / len(set(x["prompt"].split())) if len(set(x["prompt"].split())) > 0 else 0, axis=1
    )
    df["response_b_prompt_overlap"] = df.apply(
        lambda x: len(set(x["prompt"].split()).intersection(set(x["response_b"].split()))) / len(set(x["prompt"].split())) if len(set(x["prompt"].split())) > 0 else 0, axis=1
    )
    feats = feats + ["response_a_prompt_overlap", "response_b_prompt_overlap"]
    return df[feats]

In [ ]:
def transform_features(df, vectorizer):
    """
    Transform features using TfidfVectorizer and combine them.
    """
    prompt_features = vectorizer.transform(df['prompt'])
    response_a_features = vectorizer.transform(df['response_a'])
    response_b_features = vectorizer.transform(df['response_b'])
    df['prompt_len']=df['prompt'].apply(len)
    df['response_a_len']=df['response_a'].apply(len)
    df['response_b_len']=df['response_b'].apply(len)
    df['response_a_len/prompt_len']=df['response_a_len']/df['prompt_len']
    df['response_b_len/prompt_len']=df['response_b_len']/df['prompt_len']
    df['response_b_len/response_a_len']=df['response_b_len']/df['response_a_len']

    
    # Combine features and add difference features
    return np.hstack([
        prompt_features.toarray(),
        response_a_features.toarray(),
        response_b_features.toarray(),
        np.abs(response_a_features.toarray() - response_b_features.toarray()),
        df[['prompt_len','response_a_len','response_b_len', 'response_a_len/prompt_len'
           ,'response_b_len/prompt_len', 'response_b_len/response_a_len']].to_numpy(),
        compute_feats(df).to_numpy()
    ])

In [ ]:
train = pd.concat([train, train_inv])
train_features = transform_features(train, tfidf)
# train_features_inv = transform_features(train_inv, tfidf)
val_features = transform_features(valid, tfidf)
test_features  = transform_features(test, tfidf)
test_inv_features = transform_features(test_inv, tfidf)
y_train = (train["winner"]=="model_a").astype("int")
y_val = (valid["winner"]=="model_a").astype("int")

In [ ]:
param = {
        'tree_method':'auto',  # this parameter means using the GPU when training our model to speedup the training process
        'device' : 'cuda',
        'n_estimators': 2000,
        'early_stopping_rounds': 100,
        'eval_metric' : 'logloss',
        'objective' : 'binary:logistic',
        'lambda': 6.4786798596508, 'alpha': 0.030943844008921396, 'colsample_bytree': 0.5, 'subsample': 0.8, 'learning_rate': 0.018, 'max_depth': 9, 'random_state': 2024, 'min_child_weight': 207
    }
xgb_model = XGBClassifier(**param)  
xgb_model.fit(train_features, y_train ,eval_set=[(val_features,y_val)],verbose=False)

In [ ]:
from lightgbm import early_stopping,log_evaluation,LGBMClassifier


light_model = LGBMClassifier(n_estimators=1000,  
                        learning_rate=0.1,
                        early_stopping_rounds=15)  

light_model.fit(train_features, y_train,eval_set=[(val_features, y_val)], eval_metric='binary_logloss')

In [ ]:
from catboost import CatBoostClassifier
catboost_params = {'depth': 8, 'iterations': 500, 'l2_leaf_reg': 3, 'learning_rate': 0.1}
catboost_model = CatBoostClassifier(
    **catboost_params, 
    random_state=42, 
    task_type='GPU'
)
catboost_model.fit(train_features, y_train,eval_set=[(val_features, y_val)], early_stopping_rounds=50, verbose=0)

In [ ]:
a = (xgb_model.predict_proba(test_features) + light_model.predict_proba(test_features) + catboost_model.predict_proba(test_features))/3
a_inv = (xgb_model.predict_proba(test_inv_features) + light_model.predict_proba(test_inv_features) + catboost_model.predict_proba(test_inv_features))/3
prob_a, prob_b = (a[:,1]  + a_inv[:,0])/2 ,  (a[:,0]  + a_inv[:,1])/2
# test["winner"]= a_final

# test["winner"]=test["winner"].apply(lambda x: "model_a" if x==1 else "model_b")
# submission =test[["id","winner"]]
# submission.head()

In [ ]:
import os
# IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

df = pd.read_parquet("test.parquet")
preds = np.load("prob_m0.npy")
preds = preds[:,0:2]
preds = preds/preds.sum(axis = 1).reshape(-1,1)

winners = []
for i in range(len(preds)):
    winners.append("model_a" if  (0.8 * preds[i, 0] + 0.2* prob_a[i]) > (0.8 * preds[i, 1] + 0.2 * prob_b[i]) else "model_b")

sub = pd.DataFrame({
    "id": df["id"],
    "winner": winners
})
sub.to_csv("submission.csv", index=False)
print(sub.head())